# Feature Extraction for HCM Rental Room Price Prediction

This notebook extracts features from:
1. **Description text** - Binary, categorical, and numeric features
2. **Address text** - Street, ward, district, and area group
3. **Title text** - Additional context for amenities

**Data**: Rental room listings in Ho Chi Minh City with Vietnamese text

In [1]:
import pandas as pd
import numpy as np
import re
from typing import Dict, List, Optional, Tuple

## 1. Utility Functions for Text Processing

In [2]:
# Centralized Configuration
FEATURE_CONFIG = {
    'binary': {
        'has_aircon': ['máy lạnh', 'điều hòa', 'điều hoà', 'aircon', 'air conditioner'],
        'has_kitchen': ['bếp', 'bếp riêng', 'bếp chung', 'khu bếp', 'bep'],
        'has_fridge': ['tủ lạnh', 'tu lanh', 'tủ lạnh'],
        'has_wardrobe': ['tủ quần áo', 'tu quan ao', 'tủ đồ', 'tủ áo'],
        'has_bed': ['giường', 'giuong'],
        'has_mattress': ['nệm', 'nem', 'đệm'],
        'has_sofa': ['sofa', 'ghế sofa', 'salon'],
        'has_full_furniture': [
            'nội thất đầy đủ', 'full nội thất', 'đầy đủ nội thất', 
            'full nt', 'nội thất', 'nội thất đẹp', 'full đồ'
        ],
        'has_loft_or_duplex': ['gác', 'gác lửng', 'gac lung', 'duplex', 'gác xép', 'gac'],
        'has_toilet_in_room': [
            'wc riêng', 'nhà vệ sinh riêng', 'toilet riêng', 
            'vs riêng', 'nhà vệ sinh trong phòng', 'wc trong phòng'
        ],
        'has_window': ['cửa sổ', 'cua so', 'cửa kính'],
        'has_camera': ['camera', 'camera an ninh'],
        'has_security_or_guard': [
            'an ninh', 'bảo vệ', 'bao ve', 'khu an ninh', 
            'bảo đảm an ninh', 'bảo vệ 24/7'
        ],
        'has_free_time': [
            'giờ giấc tự do', 'tu do gio giac', 'không giới hạn giờ',
            'tự do', 'không chung chủ', 'riêng biệt'
        ],
        'near_school': [
            'gần trường', 'gần đại học', 'gần đh', 'gần trường học',
            'cách trường', 'gần trường đại học'
        ],
        'near_market': ['gần chợ', 'cách chợ', 'gần khu chợ'],
        'near_supermarket': [
            'gần siêu thị', 'gần coopmart', 'gần big c', 'vinmart',
            'aeon', 'tttm', 'siêu thị', 'lotte'
        ],
        'has_wifi': ['wifi', 'wi-fi', 'internet', 'mạng miễn phí', 'internet miễn phí'],
        'has_parking': ['chỗ để xe', 'bãi xe', 'giữ xe', 'đậu xe', 'gửi xe'],
        'has_balcony': ['ban công', 'ban cong', 'balcony'],
    },
    'categorical': {
        'washing_machine_type': {
            'private': ['máy giặt riêng', 'máy giặt trong phòng', 'máy giặt dùng riêng'],
            'shared': ['máy giặt chung', 'dùng chung máy giặt', 'may giat chung']
        },
        'toilet_type': {
            'private': ['wc riêng', 'toilet riêng', 'nhà vệ sinh riêng', 'vs riêng'],
            'shared': ['wc chung', 'toilet chung', 'nhà vệ sinh chung', 'vs chung']
        },
        'lock_type': {
            'fingerprint': ['khóa vân tay', 'khoa van tay', 'vân tay'],
            'card': ['khóa từ', 'khoa tu', 'thẻ từ', 'the tu'],
            'none': ['không khóa', 'không có khóa']
        },
        'water_type': {
            'tap': ['nước máy', 'nuoc may'],
            'well': ['nước giếng', 'nuoc gieng'],
            'bottled': ['nước bình', 'nước lọc', 'ro', 'nuoc binh']
        }
    },
    'price_keywords': {
        'electric_price': ['điện', 'dien', 'tiền điện', 'giá điện', 'gia dien'],
        'water_price': ['nước', 'nuoc', 'tiền nước', 'giá nước', 'gia nuoc'],
        'garbage_price': ['rác', 'rac', 'tiền rác', 'phí rác', 'phi rac'],
        'service_price': [
            'dịch vụ', 'dich vu', 'phí dv', 'phi dv', 'phí quản lý',
            'phi quan ly', 'ql', 'phí chung', 'phi chung'
        ]
    }
}

def normalize_text(text: str) -> str:
    """Normalize Vietnamese text for matching: lowercase + strip"""
    if pd.isna(text):
        return ""
    return str(text).lower().strip()

def create_regex_pattern(keywords: List[str]) -> str:
    """Create regex pattern from keywords with simple negation lookbehind"""
    escaped_kws = [re.escape(k) for k in keywords]
    kw_pattern = '|'.join(escaped_kws)
    return fr'(?<!không\s)(?<!ko\s)(?:{kw_pattern})'

def extract_price_pattern_optimized(text: str, keywords: List[str]) -> Optional[float]:
    """Extract price near keywords using regex patterns with validation."""
    if not isinstance(text, str): 
        return None
    text_norm = normalize_text(text)
    
    for kw in keywords:
        kw_pos = text_norm.find(kw)
        if kw_pos == -1: 
            continue
            
        # Search in a window around the keyword
        window = text_norm[max(0, kw_pos-30):min(len(text_norm), kw_pos+50)]
        
        # Pattern to match various price formats
        pattern = r'(\d+)[.,](\d+)k|(\d+)k|(\d+)[.,](\d{3})|(\d+)'
        
        matches = list(re.finditer(pattern, window))
        for match in matches:
            price = None
            if match.group(1):  # 3.5k format
                price = float(match.group(1)) * 1000 + float(match.group(2)) * 100
            elif match.group(3):  # 3k format
                price = float(match.group(3)) * 1000
            elif match.group(4):  # 3.000 format
                price = float(match.group(4) + match.group(5))
            elif match.group(6):  # plain number
                num = float(match.group(6))
                # Only multiply by 1000 if number is small (likely in k format)
                price = num * 1000 if num < 100 else num
            
            # Validate price range (reasonable utility prices in VND)
            if price is not None:
                # Electric: 2k-6k, Water: 10k-200k, Garbage: 10k-50k, Service: 50k-500k
                if 1000 <= price <= 1000000:  # Between 1k and 1M VND
                    return price
    
    return None


## 2. Binary Features from Description

In [3]:
def extract_boolean_features_vectorized(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    """Extract binary features using vectorized regex"""
    features_df = pd.DataFrame(index=df.index)
    
    for feature_name, keywords in FEATURE_CONFIG['binary'].items():
        pattern = create_regex_pattern(keywords)
        features_df[feature_name] = df[text_col].str.contains(pattern, regex=True, na=False)
        
    return features_df

## 3. Categorical Features from Description

In [4]:
def extract_categorical_features_vectorized(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    """Extract categorical features using vectorized regex"""
    features_df = pd.DataFrame(index=df.index)
    
    for category, type_map in FEATURE_CONFIG['categorical'].items():
        # Default to None (NaN) instead of 'unknown'
        features_df[category] = None
        
        for type_name, keywords in type_map.items():
            pattern = create_regex_pattern(keywords)
            mask = df[text_col].str.contains(pattern, regex=True, na=False)
            features_df.loc[mask, category] = type_name
            
    return features_df

## 4. Numeric Price Features from Description

In [5]:
def extract_numeric_prices_optimized(df: pd.DataFrame, text_col: str) -> pd.DataFrame:
    """Extract utility prices"""
    features_df = pd.DataFrame(index=df.index)
    
    for price_name, keywords in FEATURE_CONFIG['price_keywords'].items():
        features_df[price_name] = df[text_col].apply(
            lambda x: extract_price_pattern_optimized(x, keywords)
        )
        
    return features_df

## 5. Address Features Extraction

In [6]:
def map_district_to_area(district: Optional[str]) -> Optional[str]:
    if pd.isna(district) or not district: return None
    d = district.lower()
    
    # Center districts
    if any(x in d for x in ['quận 1', 'quận 3', 'quận 4', 'quận 5']) and '12' not in d:
        return 'center'
    
    # Near center
    if any(x in d for x in ['quận 10', 'quận 11', 'tân bình', 'phú nhuận', 'bình thạnh']):
        return 'near_center'
    
    # East
    if any(x in d for x in ['quận 2', 'quận 9', 'thủ đức']):
        return 'east'
    
    # West
    if any(x in d for x in ['quận 6', 'bình tân', 'tân phú']):
        return 'west'
    
    # South
    if any(x in d for x in ['quận 7', 'quận 8', 'nhà bè']):
        return 'south'
    
    # Outer
    if any(x in d for x in ['quận 12', 'hóc môn', 'củ chi', 'bình chánh', 'cần giờ', 'gò vấp']):
        return 'outer'
    
    return 'unknown'

def extract_address_features_vectorized(df: pd.DataFrame, address_col: str) -> pd.DataFrame:
    """Extract address features using vectorized regex"""
    if address_col not in df.columns:
        return pd.DataFrame({
            'street_name': None, 'ward': None, 'district': None, 'area_group': None
        }, index=df.index)
        
    addr_series = df[address_col].astype(str)
    features_df = pd.DataFrame(index=df.index)
    
    # Street: Capture "Đường ..."
    features_df['street_name'] = addr_series.str.extract(r'(Đường\s+[^,]+)', flags=re.IGNORECASE)[0].str.strip()
    
    # Ward: Capture "Phường ..." or "P. ..."
    ward_raw = addr_series.str.extract(r'((?:Phường|P\.|P)\s+[^,]+)', flags=re.IGNORECASE)[0].str.strip()
    # Normalize P. -> Phường
    features_df['ward'] = ward_raw.str.replace(r'^(P\.|P)\s+', 'Phường ', regex=True, flags=re.IGNORECASE)
    
    # District: Capture "Quận ..." or "Q. ..."
    district_raw = addr_series.str.extract(r'((?:Quận|Q\.|Q)\s+[^,]+)', flags=re.IGNORECASE)[0].str.strip()
    # Normalize Q. -> Quận
    features_df['district'] = district_raw.str.replace(r'^(Q\.|Q)\s+', 'Quận ', regex=True, flags=re.IGNORECASE)
    
    # Clean district (remove parens)
    features_df['district'] = features_df['district'].str.replace(r'\s*\(.*?\)', '', regex=True).str.strip()
    
    # Map area group
    features_df['area_group'] = features_df['district'].apply(map_district_to_area)
    
    return features_df

## 6. Apply All Extractions to DataFrame

In [7]:
def apply_all_feature_extraction(df: pd.DataFrame) -> pd.DataFrame:
    """Apply all feature extraction functions"""
    print("Starting feature extraction (Optimized)...")
    print(f"Original shape: {df.shape}")
    
    print("Pre-processing text...")
    df['combined_text'] = (df.get('description', '').fillna('') + ' ' + df.get('title', '').fillna('')).astype(str).str.lower()
    
    print("1. Extracting binary features...")
    boolean_df = extract_boolean_features_vectorized(df, 'combined_text')
    
    print("2. Extracting categorical features...")
    categorical_df = extract_categorical_features_vectorized(df, 'combined_text')
    
    print("3. Extracting utility prices...")
    price_df = extract_numeric_prices_optimized(df, 'combined_text')
    
    print("4. Extracting address features...")
    address_df = extract_address_features_vectorized(df, 'address')
    
    df_enriched = pd.concat([df, boolean_df, categorical_df, price_df, address_df], axis=1)
    df_enriched.drop(columns=['combined_text'], inplace=True)
    
    print(f"\nFinal shape: {df_enriched.shape}")
    print(f"Added {df_enriched.shape[1] - df.shape[1]} new features")
    
    return df_enriched

## 7. Export Enriched Data 

Once satisfied with the extracted features, you can save the enriched dataframe:

In [8]:
df = pd.read_csv('../../data/processed/data.csv')
extracted_df=apply_all_feature_extraction(df)
extracted_df.to_csv('../../data/processed/extracted_data.csv', index=False)

Starting feature extraction (Optimized)...
Original shape: (34687, 7)
Pre-processing text...
1. Extracting binary features...
2. Extracting categorical features...
2. Extracting categorical features...
3. Extracting utility prices...
3. Extracting utility prices...
4. Extracting address features...
4. Extracting address features...

Final shape: (34687, 39)
Added 31 new features

Final shape: (34687, 39)
Added 31 new features
